In [22]:
import pandas as pd
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

import pandas as pd
import plotly.graph_objects as go

pd.set_option('display.max_columns', None)

In [23]:
# Criação da sessão Spark local
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g") 
    .master("local[*]")
    .appName("threat_analysis")
    .getOrCreate()
    )

In [24]:
# caminho pra pasta com dados
data_folder_path = Path().resolve().parent.parent / "data"

## Validação da Ameaça

In [25]:
# base de ameaça criada
threat_dataset_path = str(data_folder_path / "threat_dataset")
df_threat = spark.read.parquet(threat_dataset_path)

In [26]:
# ver distribuição das 3 variáveis para ver se o min max faz sentido
# progression_distance = aumenta sempre que vai pra direita (correlação positiva e significa q está próximo do gol)
# separar alguns eventos e plotar um scatterplot dos jogadores casa/fora (cores diferentes) e uma reta vertical tracejada 
# e contar se está certo para avaliar o "total_players" e "Atk_def_advantage"
# total_players = jogadores à esquerda da linha tracejada não entram e o max deve ser 22
# atk_def_advantage = + atacantes -> positivo, - atacantes -> negativo

In [27]:
def plot_threat_event(df_threat, show_player_names=True):
    """
    Plota um evento para validar:
    - attackers_between_ball_goal
    - defenders_between_ball_goal
    - total_players_between_ball_goal
    - atk_def_advantage_between_ball_goal

    Espera um DataFrame Spark contendo exatamente um evento.
    """

    pdf = df_threat.toPandas()

    if len(pdf) != 1:
        raise ValueError("O dataframe deve conter exatamente um evento.")

    row = pdf.iloc[0]

    attackers = row["attackingPlayersNorm"]
    defenders = row["defendingPlayersNorm"]
    ball = row["ballsNorm"][0]

    stadium_length = row["stadiumLength"]
    stadium_width = row["stadiumWidth"]

    left_x = -stadium_length / 2
    right_x = stadium_length / 2
    bottom_y = -stadium_width / 2
    top_y = stadium_width / 2

    ball_x = ball["x"]

    fig = go.Figure()

    # ============================
    # Atacantes
    # ============================

    fig.add_trace(
    go.Scatter(
        x=[p["x"] for p in attackers],
        y=[p["y"] for p in attackers],
        mode="markers+text" if show_player_names else "markers",
        text=[p["player"]["name"] for p in attackers] if show_player_names else None,
        textposition="top center",
        marker=dict(
            size=10,
            color=[
                "red" if p["x"] >= ball_x else "lightcoral"
                for p in attackers
            ]
        ),
        name="Attackers"
    )
)

    # ============================
    # Defensores
    # ============================

    fig.add_trace(
    go.Scatter(
        x=[p["x"] for p in defenders],
        y=[p["y"] for p in defenders],
        mode="markers+text" if show_player_names else "markers",
        text=[p["player"]["name"] for p in defenders] if show_player_names else None,
        textposition="top center",
        marker=dict(
            size=10,
            color=[
                "blue" if p["x"] >= ball_x else "lightblue"
                for p in defenders
            ]
        ),
        name="Defenders"
    )
)
    
    # ============================
    # Bola
    # ============================

    fig.add_trace(
        go.Scatter(
            x=[ball["x"]],
            y=[ball["y"]],
            mode="markers",
            marker=dict(
                color="black",
                size=10,
                symbol="circle"
            ),
            name="Ball"
        )
    )

    # ============================
    # Linha da bola
    # ============================

    fig.add_vline(
        x=ball_x,
        line_dash="dash",
        line_width=2,
        line_color="black"
    )

    # ============================
    # Limites do campo
    # ============================

    fig.update_xaxes(
        range=[left_x, right_x],
        title="X",
        zeroline=False
    )

    fig.update_yaxes(
        range=[bottom_y, top_y],
        title="Y",
        scaleanchor="x",
        scaleratio=1,
        zeroline=False
    )
    
    # ============================
    # Contorno do campo
    # ============================

    fig.add_shape(
        type="rect",
        x0=left_x,
        y0=bottom_y,
        x1=right_x,
        y1=top_y,
        line=dict(
            color="black",
            width=2
        ),
        fillcolor="rgba(0,0,0,0)"
    )

    # ============================
    # Layout
    # ============================
    height = 600
    width = int(height * stadium_length / stadium_width)
    
    fig.update_layout(
    template="simple_white",
    width=width,
    height=height,
    margin=dict(l=20, r=120, t=60, b=20),
    title=(
        f"Attacking: {row['eventTeamName']} | "
        #f"attackingDirection: {row['attackingDirection']} | "       
        #f"Attackers: {row['attackers_between_ball_goal']} | "
        #f"Defenders: {row['defenders_between_ball_goal']} | "
        f"Total: {row['total_players_between_ball_goal']} | "
        f"Advantage: {row['atk_def_advantage_between_ball_goal']} | "
        f"Progression: {row['progression_distance']} | "
        f"Threat: {row['threat_score']:.3f}"
    ),
    legend=dict(
        x=1.02,
        y=1,
        xanchor="left",
        yanchor="top",
        orientation="v",
        bgcolor="rgba(255,255,255,0.8)"
    )
)

    fig.show()

In [28]:
df_threat.select('progression_distance').summary().show()

+-------+--------------------+
|summary|progression_distance|
+-------+--------------------+
|  count|              855540|
|   mean|  56.157468990345066|
| stddev|   23.31877998524696|
|    min|                0.09|
|    25%|               37.19|
|    50%|               54.78|
|    75%|               73.51|
|    max|              117.93|
+-------+--------------------+



In [29]:
df_threat_MC = df_threat.filter(F.col('homeTeamName') == 'Manchester City')

In [30]:
df_threat_MC.select('progression_distance').summary().show()

+-------+--------------------+
|summary|progression_distance|
+-------+--------------------+
|  count|               46537|
|   mean|    56.7902239078582|
| stddev|   23.11132122723732|
|    min|                0.58|
|    25%|               37.72|
|    50%|               55.91|
|    75%|               74.11|
|    max|              114.38|
+-------+--------------------+



In [31]:
df_threat_MC.select('date', 
                    'gameId', 
                    'startFormattedGameClock',
                    'eventId', 
                    'eventTypeDescription',                    
                    'homeTeam', 
                    'HomeTeamName',
                    'opponentTeamName',
                    'attackers_between_ball_goal',
                    'defenders_between_ball_goal',
                    'progression_distance',
                    'total_players_between_ball_goal',
                    'atk_def_advantage_between_ball_goal',
                    'progression_distance_norm',
                    'total_players_between_ball_goal_norm',
                    'atk_def_advantage_between_ball_goal_norm',
                    'threat_score'
                    ).show(1000, truncate=False)

+----------+------+-----------------------+--------------------------------+--------------------------------------+--------+---------------+-----------------+---------------------------+---------------------------+--------------------+-------------------------------+-----------------------------------+-------------------------+------------------------------------+----------------------------------------+------------+
|date      |gameId|startFormattedGameClock|eventId                         |eventTypeDescription                  |homeTeam|HomeTeamName   |opponentTeamName |attackers_between_ball_goal|defenders_between_ball_goal|progression_distance|total_players_between_ball_goal|atk_def_advantage_between_ball_goal|progression_distance_norm|total_players_between_ball_goal_norm|atk_def_advantage_between_ball_goal_norm|threat_score|
+----------+------+-----------------------+--------------------------------+--------------------------------------+--------+---------------+-----------------+

In [32]:
# chute do time da casa
event_id = "3a725c404b084914d6f1fef150fb77f9"

df_threat.filter(F.col("eventId") == event_id).show(truncate=False)

plot_threat_event(df_threat.filter(F.col("eventId") == event_id))

# attacking players e defending correto mas está atacando pra esquerda

+------+--------------------------------+---------+--------------------+------+-----------------------+--------------+--------+---------------+---------------+-------------+---------------+----------+---------+-------------+---------------+----------------+-----------------+---------------------+--------------+-------------+------------+-------------+------------------+----------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [33]:
# clearence do time adversário
event_id = 'c9cf2aa379ba29c9b804597e70e7a202'

df_threat.filter(F.col("eventId") == event_id).show(truncate=False)

plot_threat_event(df_threat.filter(F.col("eventId") == event_id))

# attacking e defending team estão corretos mas está atacando pra esquerda

+------+--------------------------------+---------+--------------------+------+-----------------------+--------------+--------+---------------+---------------+-------------+---------------+----------+---------+-------------+---------------+----------------+-----------------+---------------------+--------------+-------------+------------+-------------+------------------+----------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [34]:
# cross na posse do time adversário
event_id = '05295389fbfeb8cb3226f4610b8ca26a'

df_threat.filter(F.col("eventId") == event_id).show(truncate=False)

plot_threat_event(df_threat.filter(F.col("eventId") == event_id))

+------+--------------------------------+---------+--------------------+------+-----------------------+--------------+--------+---------------+---------------+-------------+---------------+----------+---------+-------------+---------------+----------------+-----------------+---------------------+--------------+-------------+------------+-------------+------------------+----------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

#### 1.2. Preparação da base de odds

In [35]:
# base de odds da Premier League 2022-2023
match_stats_22_23_path = str(data_folder_path / "match_stats" / "PL_22_23.csv")
df_pl_match_stats_22_23 = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(match_stats_22_23_path , sep=',')

In [36]:
df_pl_match_stats_22_23_filtrado = df_pl_match_stats_22_23.select(
    F.to_date(F.col("Date"), "dd/MM/yyyy").alias("date"),
    F.col('HomeTeam').alias('homeTeamName'),
    F.col('AwayTeam').alias('opponentTeamName'),
    'FTHG',
    'FTAG',
    'FTR',
    #'HTHG',
    #'HTAG',
    #'HTR',
    'HS',
    'AS',
    'HST',
    'AST',    
    'AvgH',
    'AvgA',
    'AvgD'
).sort('date')

In [37]:
team_name_mapping = {
    "Tottenham": "Tottenham Hotspur",
    "Brighton": "Brighton & Hove Albion",
    "Man City": "Manchester City",
    "Crystal Palace": "Crystal Palace",
    "Leicester": "Leicester City",
    "Aston Villa": "Aston Villa",
    "Bournemouth": "AFC Bournemouth",
    "Fulham": "Fulham",
    "West Ham": "West Ham",
    "Man United": "Manchester United",
    "Wolves": "Wolverhampton Wanderers",
    "Southampton": "Southampton",
    "Liverpool": "Liverpool",
    "Chelsea": "Chelsea",
    "Nott'm Forest": "Nottingham Forest",
    "Newcastle": "Newcastle United",
    "Everton": "Everton",
    "Leeds": "Leeds United",
    "Arsenal": "Arsenal",
    "Brentford": "Brentford"
}

df_pl_match_stats_22_23_filtrado_mapped = (
    df_pl_match_stats_22_23_filtrado
    .replace(team_name_mapping, subset=["homeTeamName", "opponentTeamName"])
)

df_pl_match_stats_22_23_filtrado_mapped.show()

+----------+--------------------+--------------------+----+----+---+---+---+---+---+-----+-----+-----+
|      date|        homeTeamName|    opponentTeamName|FTHG|FTAG|FTR| HS| AS|HST|AST| AvgH| AvgA| AvgD|
+----------+--------------------+--------------------+----+----+---+---+---+---+---+-----+-----+-----+
|2022-08-05|      Crystal Palace|             Arsenal|   0|   2|  A| 10| 10|  2|  2| 4.39| 1.88| 3.59|
|2022-08-06|              Fulham|           Liverpool|   2|   2|  D|  9| 11|  3|  4|10.99| 1.28| 6.05|
|2022-08-06|     AFC Bournemouth|         Aston Villa|   2|   0|  H|  7| 15|  3|  2|  3.8| 2.04|  3.5|
|2022-08-06|        Leeds United|Wolverhampton Wan...|   2|   1|  H| 12| 15|  4|  6| 2.34| 3.18| 3.34|
|2022-08-06|    Newcastle United|   Nottingham Forest|   2|   0|  H| 23|  5| 10|  0| 1.67| 5.57|  3.8|
|2022-08-06|   Tottenham Hotspur|         Southampton|   4|   1|  H| 18| 10|  8|  2| 1.36| 8.64| 5.27|
|2022-08-06|             Everton|             Chelsea|   0|   1|  A|  8| 

### 2. Criação das bases agregadas e teste de correlação

### 2.1. Por Time-Partida

### 2.2. Por Time-Partida-Ciclo de Posse

In [38]:
df_threat.show()

+------+--------------------+------------+--------------------+------+-----------------------+--------------+--------+----------------+--------------+-------------+---------------+----------+---------+-------------+--------------+----------------+-----------------+---------------------+-------------+-------------+------------+-------------+------------------+----------------+--------------------+--------------------+--------------------+---------------------------+---------------------------+--------------------+-------------------------------+-----------------------------------+-------------------------+------------------------------------+----------------------------------------+------------+
|gameId|             eventId|   eventType|eventTypeDescription|period|startFormattedGameClock|startGameClock|homeTeam| eventPlayerName| eventTeamName|competitionId|competitionName|      date|   season|    venueType|  homeTeamName|opponentTeamName|homeTeamStartSide|opponentTeamStartSide|  stadium

In [39]:
df_avg_threat = (
    df_threat
    .groupBy(
        "gameId",
        "competitionId",
        "season",
        "date",
        'homeTeamName', 
        'opponentTeamName',
        "homeTeam",
    )
    .agg(
        F.round(F.mean(F.col("threat_score")), 3).alias("avg_threat_score"),
        F.round(F.mean(F.col('attackers_between_ball_goal')), 2).alias('avg_attackers_between_ball_goal'),
        F.round(F.mean(F.col('defenders_between_ball_goal')), 2).alias('avg_defenders_between_ball_goal'),
        F.round(F.mean(F.col('progression_distance')), 2).alias('avg_progression_distance'),
        F.round(F.mean(F.col('total_players_between_ball_goal')), 2).alias('avg_total_players_between_ball_goal'),
        F.round(F.mean(F.col('atk_def_advantage_between_ball_goal')), 2).alias('avg_atk_def_advantage_between_ball_goal'),
        F.round(F.mean(F.col('progression_distance_norm')), 2).alias('avg_progression_distance_norm'),
        F.round(F.mean(F.col('total_players_between_ball_goal_norm')), 2).alias('avg_total_players_between_ball_goal_norm'),
        F.round(F.mean(F.col('atk_def_advantage_between_ball_goal_norm')), 2).alias('avg_atk_def_advantage_between_ball_goal_norm'),
    )
)

df_avg_threat.show()

+------+-------------+---------+----------+--------------------+--------------------+--------+----------------+-------------------------------+-------------------------------+------------------------+-----------------------------------+---------------------------------------+-----------------------------+----------------------------------------+--------------------------------------------+
|gameId|competitionId|   season|      date|        homeTeamName|    opponentTeamName|homeTeam|avg_threat_score|avg_attackers_between_ball_goal|avg_defenders_between_ball_goal|avg_progression_distance|avg_total_players_between_ball_goal|avg_atk_def_advantage_between_ball_goal|avg_progression_distance_norm|avg_total_players_between_ball_goal_norm|avg_atk_def_advantage_between_ball_goal_norm|
+------+-------------+---------+----------+--------------------+--------------------+--------+----------------+-------------------------------+-------------------------------+------------------------+--------------

In [40]:
df_avg_threat_match_stats = (
    df_avg_threat.join(
        df_pl_match_stats_22_23_filtrado_mapped,
        on= ['date', 'homeTeamName', 'opponentTeamName'],
        how='left'
    )
)

df_avg_threat_match_stats.show()

+----------+--------------------+--------------------+------+-------------+---------+--------+----------------+-------------------------------+-------------------------------+------------------------+-----------------------------------+---------------------------------------+-----------------------------+----------------------------------------+--------------------------------------------+----+----+---+---+---+---+---+----+----+----+
|      date|        homeTeamName|    opponentTeamName|gameId|competitionId|   season|homeTeam|avg_threat_score|avg_attackers_between_ball_goal|avg_defenders_between_ball_goal|avg_progression_distance|avg_total_players_between_ball_goal|avg_atk_def_advantage_between_ball_goal|avg_progression_distance_norm|avg_total_players_between_ball_goal_norm|avg_atk_def_advantage_between_ball_goal_norm|FTHG|FTAG|FTR| HS| AS|HST|AST|AvgH|AvgA|AvgD|
+----------+--------------------+--------------------+------+-------------+---------+--------+----------------+-------------

In [41]:

df_pl_match_stats_22_23_filtrado_mapped_home = (
    df_avg_threat_match_stats
    .filter(
        F.col('homeTeam')
        )
    .select(
        'competitionId',
        'season',
        'gameId',
        "date",
        F.col("homeTeamName").alias("teamName"),
        "avg_threat_score",
        #'avg_attackers_between_ball_goal',
        #'avg_defenders_between_ball_goal',
        #'avg_progression_distance',
        #'avg_total_players_between_ball_goal',
        #'avg_atk_def_advantage_between_ball_goal',
        'avg_progression_distance_norm',
        'avg_total_players_between_ball_goal_norm',
        'avg_atk_def_advantage_between_ball_goal_norm',
        (F.col("FTR") == 'H').alias('win'),
        F.col("FTHG").alias("goals"),
        F.col("HS").alias("shots"),
        F.col("HST").alias("shots_target"),
        F.col("AvgH").alias("avg_win_odds")
    )
)

df_pl_match_stats_22_23_filtrado_mapped_away = (
    df_avg_threat_match_stats
    .filter(
        ~F.col('homeTeam')
    )
    .select(
        'competitionId',
        'season',
        'gameId',
        "date",
        F.col("opponentTeamName").alias("teamName"),
        "avg_threat_score",
        #'avg_attackers_between_ball_goal',
        #'avg_defenders_between_ball_goal',
        #'avg_progression_distance',
        #'avg_total_players_between_ball_goal',
        #'avg_atk_def_advantage_between_ball_goal',
        'avg_progression_distance_norm',
        'avg_total_players_between_ball_goal_norm',
        'avg_atk_def_advantage_between_ball_goal_norm',
        (F.col("FTR") == 'A').alias('win'),
        F.col("FTAG").alias("goals"),
        F.col("AS").alias("shots"),
        F.col("AST").alias("shots_target"),
        F.col("AvgA").alias("avg_win_odds")
))

df_team_match = df_pl_match_stats_22_23_filtrado_mapped_home.unionByName(df_pl_match_stats_22_23_filtrado_mapped_away).sort('avg_threat_score', ascending=False)
df_team_match.show(truncate=False)

+-------------+---------+------+----------+----------------------+----------------+-----------------------------+----------------------------------------+--------------------------------------------+-----+-----+-----+------------+------------+
|competitionId|season   |gameId|date      |teamName              |avg_threat_score|avg_progression_distance_norm|avg_total_players_between_ball_goal_norm|avg_atk_def_advantage_between_ball_goal_norm|win  |goals|shots|shots_target|avg_win_odds|
+-------------+---------+------+----------+----------------------+----------------+-----------------------------+----------------------------------------+--------------------------------------------+-----+-----+-----+------------+------------+
|1            |2022-2023|4613  |2023-01-01|Aston Villa           |0.541           |0.54                         |0.57                                    |0.52                                        |true |2    |13   |4           |4.94        |
|1            |2022-2023

In [42]:
remove_cols = ['competitionId','season','gameId','date','teamName']

df_team_match_pd = df_team_match.toPandas()
df_team_match_pd.drop(remove_cols, axis=1).corr()

,avg_threat_score,avg_progression_distance_norm,avg_total_players_between_ball_goal_norm,avg_atk_def_advantage_between_ball_goal_norm,win,goals,shots,shots_target,avg_win_odds
avg_threat_score,1.000000,0.369777,0.957691,0.942226,0.064332,0.016054,0.115222,0.085076,-0.049537
avg_progression_distance_norm,0.369777,1.000000,0.158860,0.112000,0.057174,0.066133,0.375581,0.189816,-0.271466
avg_total_players_between_ball_goal_norm,0.957691,0.158860,1.000000,0.915635,0.074919,0.025903,0.033546,0.057190,0.020623
avg_atk_def_advantage_between_ball_goal_norm,0.942226,0.112000,0.915635,1.000000,0.032663,-0.027781,0.021204,0.027857,0.008850
win,0.064332,0.057174,0.074919,0.032663,1.000000,0.620001,0.252019,0.385947,-0.287305
goals,0.016054,0.066133,0.025903,-0.027781,0.620001,1.000000,0.330004,0.587279,-0.224993
shots,0.115222,0.375581,0.033546,0.021204,0.252019,0.330004,1.000000,0.695799,-0.421675
shots_target,0.085076,0.189816,0.057190,0.027857,0.385947,0.587279,0.695799,1.000000,-0.306992
avg_win_odds,-0.049537,-0.271466,0.020623,0.008850,-0.287305,-0.224993,-0.421675,-0.306992,1.000000
